# 05 — The fifteen DOTA classes

Every DOTA checkpoint on the Hub speaks the same fifteen-class vocabulary. This lesson looks at that vocabulary from three angles: the class list and its order, which classes the published model finds hard, and what predictions look like next to real labels on DOTA validation tiles.

| | |
|---|---|
| Needs | Internet (Hub checkpoint); sections 3–5 need a local checkout with `demo/dota_samples/` |
| Kaggle | Sections 1–2 only. The DOTA tiles are **academic-use only** and are not mirrored to public kernels. |
| GPU | Optional (36 tiles take ~20 s on a GPU, a few minutes on CPU) |
| Posts | [Oriented R-CNN detections for the 15 DOTA classes](https://deeplearning.earth/posts/2026-06-23_oriented_rcnn_detections_for_the_15_dota_classes/) · [Apache licence versus DOTA](https://deeplearning.earth/posts/2026-09-10_oriented_det_apache_license_versus_dota/) |

## DOTA in one paragraph

DOTA v1.0 is the reference benchmark for oriented detection in aerial imagery: 2,806 images from Google Earth and Chinese satellites, 800 to 4,000 px on a side, with about 188,000 objects labelled as four-corner polygons. Images are cut into 1024×1024 tiles with 200 px overlap (`odet tile-dota`) for training and evaluation. The test labels are hidden; scores come from the organisers' server ("Task 1" for oriented boxes). The data is licensed for academic use only, which is why OrientedDet's Apache-2.0 code ships only a handful of sample tiles.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. The class list, and why order matters

`DOTA_V1_CLASSES` lists the classes in the order of the DOTA paper. A checkpoint stores its own `class_names`, which for the Hub models is **alphabetical**. The integer `label` in a detection indexes the checkpoint's list (1-based), so always map labels with the checkpoint's `class_names`, never with a hard-coded list.

In [ ]:
from oriented_det.data.dota_classes import DOTA_V1_CLASSES

model, config, class_names = hub_model("oriented_rcnn_dota_le90_1x")
print(f"{'label':>5s}  {'checkpoint class_names':24s} DOTA paper order")
for i, (ours, paper) in enumerate(zip(class_names, DOTA_V1_CLASSES), start=1):
    print(f"{i:5d}  {ours:24s} {paper}")

## 2. Which classes are hard?

The committed eval report for this checkpoint contains its official Task 1 score per class. It is read from the checkout, or downloaded from GitHub (the reports are Apache-2.0 docs, not DOTA data).

In [ ]:
import re


def eval_report(slug):
    rel = f"docs/eval-reports/{slug}/model_analysis.md"
    path = REPO / rel if HAS_REPO else WORK / rel
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/{rel}", path)
    return path.read_text(encoding="utf-8")


task1 = eval_report("oriented_rcnn_dota_le90_1x").split("## Official Task 1")[1].split("\n## ")[0]
ap = {m[1]: float(m[2]) for m in re.finditer(r"^\| ([a-z-]+) \| ([0-9.]+) \|$", task1, re.M)}
order = sorted(ap, key=ap.get)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(order, [100 * ap[c] for c in order], color="tab:blue")
ax.axvline(100 * sum(ap.values()) / len(ap), color="k", ls="--", label=f"mAP {100 * sum(ap.values()) / len(ap):.2f}%")
ax.set_xlabel("official Task 1 AP50 (%)"); ax.legend(); plt.tight_layout(); plt.show()

The hard classes share a story. **Bridges** are long, thin, and defined by context (a road over water), so their boxes are easy to get slightly wrong. **Soccer-ball fields** and **ground-track fields** look alike from above. **Roundabouts** and **harbours** have fuzzy extents that annotators draw inconsistently. **Helicopters** are rare. The easy classes (tennis courts, planes, ships) are compact, frequent, and have crisp outlines.

## 3. The sample tiles and their labels

In [ ]:
SAMPLES = REPO / "demo" / "dota_samples" if HAS_REPO else None
if SAMPLES is None or not SAMPLES.is_dir():
    SAMPLES = None
    print("demo/dota_samples/ needs a local checkout. DOTA tiles are academic-use only and are not "
          "mirrored to Kaggle, so the sections that use them are skipped.")

In [ ]:
from collections import Counter
from oriented_det.data.dota import DOTAAnnotation

tiles = {}
if SAMPLES:
    for img in sorted(SAMPLES.glob("*.jpg")):
        lines = [l for l in img.with_suffix(".txt").read_text().splitlines() if l.strip()]
        tiles[img] = [DOTAAnnotation.from_line(l) for l in lines]
    counts = Counter(a.class_name for anns in tiles.values() for a in anns)
    print(f"{len(tiles)} tiles, {sum(counts.values())} labelled objects")
    for name in class_names:
        print(f"  {counts[name]:5d}  {name}")

Class frequencies are wildly unbalanced: thousands of vehicles and ships against a handful of helicopters and roundabouts. That imbalance, repeated across the full dataset, is one reason rare classes score lower.

## 4. Predictions next to labels

Run the detector on every sample tile, then pick four tiles that together cover as many classes as possible (a greedy pick over the labels). Green: labels. Coloured with captions: predictions at the deploy floor.

In [ ]:
from PIL import Image
from oriented_det.runtime.inference import visualize_results
from oriented_det.utils.viz import DrawingSpec, draw_boxes

preds = {t: detect(model, config, class_names, Image.open(t).convert("RGB")) for t in tiles}

picked, covered = [], set()
for _ in range(4 if tiles else 0):
    best = max(tiles, key=lambda t: len({a.class_name for a in tiles[t]} - covered))
    picked.append(best)
    covered |= {a.class_name for a in tiles[best]}
if picked:
    print("tiles:", ", ".join(t.stem for t in picked), f"-> {len(covered)} classes covered")
for t in picked:
    img = Image.open(t).convert("RGB")
    gt = draw_boxes(img.copy(), [a.rbox for a in tiles[t]], specs=[DrawingSpec(outline=(0, 255, 0), width=2)] * len(tiles[t]))
    show(gt, visualize_results(img.copy(), preds[t], class_names),
         titles=(f"{t.stem}: {len(tiles[t])} labels", f"{len(preds[t])} predictions"), size=5)

## 5. Where does the model go wrong?

For every non-difficult label, find the prediction that overlaps it most (rotated IoU). If that IoU is at least 0.5 and the classes agree, it is found. If the classes disagree, it is a confusion. Otherwise it is missed.

In [ ]:
from oriented_det.ops import rbox_iou

found, total, confused = Counter(), Counter(), Counter()
for t, anns in tiles.items():
    for a in anns:
        if a.difficult:
            continue
        total[a.class_name] += 1
        iou, best = max(((rbox_iou(d["rbox"], a.rbox), d) for d in preds[t]), default=(0.0, None), key=lambda x: x[0])
        if iou >= 0.5:
            pred_name = class_names[best["label"] - 1]
            if pred_name == a.class_name:
                found[a.class_name] += 1
            else:
                confused[(a.class_name, pred_name)] += 1
if total:
    print(f"{'class':20s} {'recall':>7s}  (labels)")
    for name in sorted(total, key=lambda n: found[n] / total[n]):
        print(f"{name:20s} {found[name] / total[name]:7.2f}  ({total[name]})")
    print("\nmost common confusions (label -> prediction):")
    for (gt_name, pred_name), n in confused.most_common(5):
        print(f"  {n:3d}  {gt_name} -> {pred_name}")

These tiles come from DOTA **val**, which the Hub 1× models also trained on, so recall here is optimistic. Lesson 07 turns this into a proper mAP and explains why the official number is lower.

## What to try next

- Repeat sections 4–5 with `rotated_fcos_dota_le90_1x` and compare which classes each detector misses.
- Lower the threshold with `detect(..., score_thr=0.3)` and watch recall rise and confusions grow.
- Checkpoints can carry per-class floors in `production.per_class_score_threshold`: pick one for small vehicles from the recall table above.
- **Next:** [06 — Large images and Sentinel-2](06_large_images_sentinel2.ipynb).